In [ ]:
from flask import Flask, render_template, jsonify, request
import math

app = Flask(__name__)

# System State
state = {
    "x": 50.0, "y": 50.0,
    "vx": 0.0, "vy": 0.0,
    "integral_x": 0.0, "integral_y": 0.0,
    "last_error_x": 0.0, "last_error_y": 0.0,
    "target_x": 0.0, "target_y": 0.0,
    "mass": 1.2,
    "dt": 0.05,
    "damping_base": 0.98
}

@app.route('/')
def index():
    return render_template('index.html')

@app.route('/update', methods=['POST'])
def update():
    data = request.json
    dist_x = float(data.get('dist_x', 0))
    dist_y = float(data.get('dist_y', 0))
    
    # Coefficients from UI Drag Bars
    kp = float(data.get('kp', 10.7)) / 10.0
    kd = float(data.get('kd', 7.3)) / 10.0
    ki = float(data.get('ki', 3.0)) / 100.0

    # 1. Error e(t)
    error_x = state["target_x"] - state["x"]
    error_y = state["target_y"] - state["y"]

    # 2. Integral sum
    state["integral_x"] += error_x
    state["integral_y"] += error_y

    # 3. Derivative (Difference)
    diff_x = error_x - state["last_error_x"]
    diff_y = error_y - state["last_error_y"]

    # 4. Control Force Fc = (P + I + D)
    fc_x = (error_x * kp) + (state["integral_x"] * ki) + (diff_x * kd)
    fc_y = (error_y * kp) + (state["integral_y"] * ki) + (diff_y * kd)

    # 5. Physics Integration
    f_net_x = fc_x + dist_x
    f_net_y = fc_y + dist_y

    state["vx"] = (state["vx"] + (f_net_x / state["mass"])) * state["damping_base"]
    state["vy"] = (state["vy"] + (f_net_y / state["mass"])) * state["damping_base"]
    state["x"] += state["vx"]
    state["y"] += state["vy"]

    state["last_error_x"] = error_x
    state["last_error_y"] = error_y

    return jsonify({
        "x": state["x"], "y": state["y"],
        "fc": math.sqrt(fc_x**2 + fc_y**2),
        "error": math.sqrt(error_x**2 + error_y**2)
    })

if __name__ == '__main__':
    app.run(debug=True, port=8011, use_reloader = False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:8011
Press CTRL+C to quit
127.0.0.1 - - [13/Apr/2026 14:45:48] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - - [13/Apr/2026 14:45:48] "POST /update HTTP/1.1" 200 -
127.0.0.1 - 